# Mascot Unitree — Wan2.1 + MediaPipe Pose Tracking

**Pipeline:**
1. Wan2.1 generates a **human** performing the gesture (much better quality than 'robot')
2. MediaPipe Pose extracts 33 skeleton landmarks per frame
3. Joint angles are retargeted to G1 robot joint space
4. Video + trajectory returned to MacBook → MuJoCo drives the simulation

**GPU:** NVIDIA L4 (24GB) — `bfloat16`, no CPU offloading needed

In [ ]:
# ── STEP 1: Install all dependencies ─────────────────────────────────────────
!pip install -q diffusers transformers accelerate torch torchvision sentencepiece
!pip install -q fastapi uvicorn pyngrok nest-asyncio imageio[ffmpeg] ftfy
!pip install -q mediapipe opencv-python-headless
print('✅ All dependencies installed')

In [ ]:
# ── STEP 2: Ngrok auth ────────────────────────────────────────────────────────
NGROK_AUTH_TOKEN = "INSERT_YOUR_NGROK_TOKEN_HERE"
import os
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTH_TOKEN
!ngrok config add-authtoken $NGROK_AUTH_TOKEN
print('✅ Ngrok configured')

In [ ]:
# ── STEP 3: Load Wan2.1-T2V-1.3B-Diffusers ──────────────────────────────────
import torch
from diffusers import WanPipeline

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
print(f"Loading {MODEL_ID}... (first run ~80s from cache)")

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f"dtype={dtype} | GPU={torch.cuda.get_device_name()}")

pipe = WanPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.to("cuda")
print("✅ Wan2.1 loaded!")

In [ ]:
# ── STEP 4: Video generation + MediaPipe pose extraction ─────────────────────
import base64
import numpy as np
import imageio
import cv2
import mediapipe as mp


def generate_video(gesture_description: str, output_path: str = "output.mp4") -> str:
    """
    Generate a video of a HUMAN performing the gesture.
    Human subjects produce much higher quality results than 'humanoid robot'.
    The pose tracker will retarget to the G1 robot afterwards.
    """
    # Build prompt: describe a human, not a robot
    # Strip any 'humanoid robot' prefix the MacBook may have added
    clean_desc = gesture_description.replace("A humanoid robot standing upright performs:", "").strip()
    clean_desc = clean_desc.replace("humanoid robot", "person").strip()

    full_prompt = (
        f"A person performs the following action: {clean_desc}. "
        "Full body visible, studio lighting, static camera, eye-level perspective, "
        "standing on a flat floor, both feet planted."
    )
    negative_prompt = (
        "robot, android, multiple people, crowd, "
        "jumping, flying, blur, watermark, text"
    )

    print(f"🎬 Prompt: {full_prompt[:100]}...")

    output = pipe(
        prompt=full_prompt,
        negative_prompt=negative_prompt,
        num_frames=49,           # ~2 sec @ 24fps — sweet spot for quality vs speed
        num_inference_steps=30,
        guidance_scale=6.0,
        height=480,
        width=832,
        generator=torch.Generator(device="cuda").manual_seed(42),
    )

    frames = output.frames[0]  # List of PIL images

    # ✅ Fix: Wan2.1 returns float32 [0,1] — scale to uint8 [0,255]
    frames_np = [(np.array(f) * 255).astype(np.uint8)
                 if np.array(f).max() <= 1.0
                 else np.array(f).astype(np.uint8)
                 for f in frames]

    imageio.mimsave(output_path, frames_np, fps=24, codec="libx264")
    print(f"✅ Video saved: {output_path}")
    return output_path


def extract_pose_trajectory(video_path: str) -> list:
    """
    Run MediaPipe Pose on the generated human video.
    Returns a list of dicts (one per frame) mapping G1 joint names → angle (radians).
    """
    mp_pose = mp.solutions.pose
    pose_detector = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)
    trajectory = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = pose_detector.process(rgb)

        if result.pose_landmarks:
            lm = result.pose_landmarks.landmark
            frame_pose = _landmarks_to_g1_joints(lm)
            trajectory.append(frame_pose)

    cap.release()
    pose_detector.close()
    print(f"✅ Pose extracted: {len(trajectory)} frames")
    return trajectory


def _landmarks_to_g1_joints(lm) -> dict:
    """
    Retarget MediaPipe 33-landmark body to G1 upper-body joints.
    MediaPipe indices: 11=left_shoulder, 12=right_shoulder,
                       13=left_elbow, 14=right_elbow,
                       15=left_wrist, 16=right_wrist,
                       23=left_hip, 24=right_hip
    """
    def vec_angle(a, b):
        """Angle of vector from landmark a to landmark b (in radians)."""
        dx = b.x - a.x
        dy = -(b.y - a.y)  # Invert Y: MediaPipe Y grows downward
        return float(np.arctan2(dy, dx))

    def mid(a, b):
        """Midpoint landmark (used for torso tilt)."""
        class M:
            x = (a.x + b.x) / 2
            y = (a.y + b.y) / 2
        return M()

    # Shoulder → elbow defines shoulder pitch
    # Elbow → wrist defines elbow flexion
    # Clamp all joint angles to physically safe ranges
    def clamp(v, lo=-1.5, hi=1.5):
        return max(lo, min(hi, v))

    rsp = clamp(vec_angle(lm[12], lm[14]))  # right shoulder pitch
    rej = clamp(vec_angle(lm[14], lm[16]))  # right elbow
    lsp = clamp(vec_angle(lm[11], lm[13]))  # left shoulder pitch
    lej = clamp(vec_angle(lm[13], lm[15]))  # left elbow
    torso = clamp(vec_angle(mid(lm[11], lm[12]), mid(lm[23], lm[24])), -0.5, 0.5)

    return {
        "right_shoulder_pitch_joint": rsp,
        "right_shoulder_roll_joint":  clamp(rsp * 0.3),  # Approximated from pitch
        "right_elbow_joint":          rej,
        "left_shoulder_pitch_joint":  lsp,
        "left_shoulder_roll_joint":   clamp(lsp * 0.3),
        "left_elbow_joint":           lej,
        "waist_yaw_joint":            torso,
    }


def video_to_base64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


print("✅ Video generation + pose tracking functions ready")

In [ ]:
# ── STEP 5: FastAPI server + Ngrok tunnel ─────────────────────────────────────
import nest_asyncio
import asyncio
import uvicorn
from pyngrok import ngrok
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()
app = FastAPI(title="Mascot Unitree — Wan2.1 + Pose Tracking")


class GestureRequest(BaseModel):
    gesture_description: str


@app.get("/")
async def health():
    return {"status": "online", "model": "Wan2.1-T2V-1.3B + MediaPipe"}


@app.post("/generate_motion")
async def generate_motion(req: GestureRequest):
    print(f"\n📨 Received: {req.gesture_description[:100]}")

    # 1. Generate human video with Wan2.1
    vid_path = generate_video(req.gesture_description)

    # 2. Extract pose trajectory via MediaPipe
    trajectory = extract_pose_trajectory(vid_path)

    # 3. Return video + trajectory to MacBook
    print(f"📤 Sending: video + {len(trajectory)}-frame trajectory")
    return {
        "status": "success",
        "model": "Wan2.1-T2V-1.3B + MediaPipe Retargeting",
        "video_base64": video_to_base64(vid_path),
        "trajectory": trajectory   # ← list of {joint_name: angle} dicts, one per frame
    }


# Start ngrok tunnel
public_url = ngrok.connect(8000)
print("="*60)
print(f"🚀 WAN2.1 + POSE TRACKING SERVER IS LIVE!")
print(f"   Copy this URL into your MacBook React dashboard:")
print(f"   {public_url.public_url}")
print("="*60)

# Use await instead of uvicorn.run() for Colab compatibility
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()